# Parent Document Retriever [Step 2 - Search Small, Return Big]

> **MLCourse - Agentic AI - Advanced RAG - Contextual Retrieval**

LangChain's `ParentDocumentRetriever` implements the small-to-big fix directly,
and it is the one to reach for by default.

The architecture is two stores that point at each other:

```
   +----------------------------+        +--------------------------+
   |  VECTOR STORE              |        |  DOC STORE               |
   |  small child chunks        |        |  full parent documents   |
   |  each with parent_id  -----+------->|  keyed by parent_id      |
   |  -> sharp embeddings       |        |  -> complete context     |
   +----------------------------+        +--------------------------+

   query -> search children -> collect their parent_ids -> return parents
```

You get the retrieval precision of a 200-character chunk and the answering power
of a full paragraph, with no compromise chunk size to tune.

### 1. Setup


In [1]:
import os
import re
import time
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
from dotenv import load_dotenv


def find_env(start=None):
    """Walk up from the notebook directory until a .env file appears."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        candidate = folder / ".env"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("No .env found walking up from " + str(start))


ENV_PATH = find_env()
load_dotenv(ENV_PATH)
DATA_DIR = ENV_PATH.parent / "data"

print("env file :", ENV_PATH)
print("data dir :", DATA_DIR)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))

env file : D:\projects\python\MLCourse\03_agentic_ai\.env
data dir : D:\projects\python\MLCourse\03_agentic_ai\data
GROQ_API_KEY present: True


In [2]:
from langchain_groq import ChatGroq

GROQ_MODEL = "qwen/qwen3.8-27b"          # verified available on this account
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)


def clean(text):
    """Strip any <think>...</think> block a reasoning model may emit."""
    return THINK_RE.sub("", text).strip()


def ask(prompt, retries=4, pause=1.5):
    """Call Groq with exponential backoff. Free tier is roughly 8000 tokens/minute,
    so every loop in these notebooks paces itself and retries on rate limits."""
    delay = 5.0
    for attempt in range(retries):
        try:
            answer = clean(llm.invoke(prompt).content)
            time.sleep(pause)
            return answer
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print(f"  [retry {attempt + 1}] {type(exc).__name__} - sleeping {delay:.0f}s")
            time.sleep(delay)
            delay *= 2


print("Groq model:", GROQ_MODEL)
print("smoke test:", ask("Reply with exactly one word: ready"))

Groq model: qwen/qwen3.8-27b


smoke test: ready


In [3]:
ALICE_PATH = DATA_DIR / "alice.txt"
raw_text = ALICE_PATH.read_text(encoding="utf-8-sig")

# "Parent" units: paragraphs. Big enough to answer from, too big to retrieve
# precisely. These are the documents we will later cut into small children.
parents = [" ".join(p.split()) for p in raw_text.split("\n\n") if len(p.strip()) > 200]

print("parent paragraphs:", len(parents))
print("mean parent length:", int(sum(len(p) for p in parents) / len(parents)), "chars")

parent paragraphs: 237
mean parent length: 379 chars


### 2. Building it by hand first

Before using the LangChain class, build the mechanism in twenty lines. It is
simple enough that seeing it explicitly makes the class configuration obvious
afterwards.

In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def build_index(texts):
    """Embed a list of texts and return the normalised matrix."""
    return encoder.encode(texts, normalize_embeddings=True,
                          batch_size=64, show_progress_bar=False)


def search(index, texts, query, top_n=5):
    """Return [(position, score)] of the best matches in `index`."""
    q = encoder.encode([query], normalize_embeddings=True)[0]
    sims = index @ q
    order = np.argsort(sims)[::-1][:top_n]
    return [(int(i), float(sims[i])) for i in order]


print("encoder ready:", encoder.get_sentence_embedding_dimension(), "dimensions")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

encoder ready: 384 dimensions


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

child_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)

child_texts, child_parent = [], []
for parent_id, parent in enumerate(parents):
    for piece in child_splitter.split_text(parent):
        child_texts.append(piece)
        child_parent.append(parent_id)

child_vectors = build_index(child_texts)

print(f"{len(parents)} parents -> {len(child_texts)} children "
      f"({len(child_texts) / len(parents):.1f} children per parent)")


def manual_parent_retrieve(query, k_children=8, max_parents=3):
    """Search children, then return their DISTINCT parents in child-rank order."""
    hits = search(child_vectors, child_texts, query, top_n=k_children)
    seen, out = set(), []
    for pos, score in hits:
        pid = child_parent[pos]
        if pid in seen:
            continue
        seen.add(pid)
        out.append((pid, child_texts[pos], score))
        if len(out) == max_parents:
            break
    return out


QUESTION = "What did the Dormouse say about the treacle well?"
for pid, child, score in manual_parent_retrieve(QUESTION):
    print(f"parent_{pid}  (matched child cosine={score:.3f})")
    print(f"  matched child : {child[:110]}...")
    print(f"  returned parent ({len(parents[pid])} chars): {parents[pid][:110]}...")
    print()

237 parents -> 622 children (2.6 children per parent)
parent_136  (matched child cosine=0.553)
  matched child : The Dormouse had closed its eyes by this time, and was going off into a doze; but, on being pinched by the Hat...
  returned parent (359 chars): The Dormouse had closed its eyes by this time, and was going off into a doze; but, on being pinched by the Hat...

parent_126  (matched child cosine=0.500)
  matched child : the other two were using it as a cushion, resting their elbows on it, and talking over its head. “Very uncomfo...
  returned parent (374 chars): There was a table set out under a tree in front of the house, and the March Hare and the Hatter were having te...

parent_137  (matched child cosine=0.460)
  matched child : least notice of her going, though she looked back once or twice, half hoping that they would call after her: t...
  returned parent (361 chars): This piece of rudeness was more than Alice could bear: she got up in great disgust, and walked off; the 

Two details in that loop matter more than they look:

**Deduplication.** Several children of the same parent often match the same
query. Without the `seen` set you would return the same paragraph three times
and fill the context window with duplicates.

**Rank inheritance.** The parents come back ordered by their *best* child's
score. That is the standard choice and it works well; an alternative is to score
parents by how many children matched, which favours broadly-relevant parents
over narrowly-relevant ones.

### 3. The LangChain implementation

`ParentDocumentRetriever` bundles exactly this. Its three moving parts:

- `vectorstore` - holds the child chunks and their `doc_id` metadata.
- `docstore` - a key-value store (`InMemoryStore` here; Redis, a filesystem or a
  database in production) holding the parents.
- `child_splitter` - how parents get cut into children.

Note the import path: in LangChain 1.x these retrievers live in
`langchain_classic`, while `InMemoryStore` comes from `langchain_core.stores`.

In [6]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma(collection_name="alice_children", embedding_function=embeddings)
docstore = InMemoryStore()

retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30),
    search_kwargs={"k": 8},
)

parent_docs = [Document(page_content=p, metadata={"parent_id": i})
               for i, p in enumerate(parents)]
retriever.add_documents(parent_docs)

print("children indexed:", vectorstore._collection.count())
print("parents stored  :", len(list(docstore.yield_keys())))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

children indexed: 622
parents stored  : 237


`add_documents` did three things in one call: split each parent into children,
embedded and stored the children with a `doc_id` pointing back, and put the
parents in the docstore. That is the entire setup cost.

### 4. Retrieval returns parents, not children


In [7]:
results = retriever.invoke(QUESTION)

print(f"query: {QUESTION}")
print(f"returned {len(results)} PARENT documents\n")
for i, doc in enumerate(results[:3], 1):
    print(f"#{i} parent_id={doc.metadata.get('parent_id')} "
          f"({len(doc.page_content)} chars)")
    print("   ", doc.page_content[:170], "...")
    print()

query: What did the Dormouse say about the treacle well?
returned 7 PARENT documents

#1 parent_id=136 (359 chars)
    The Dormouse had closed its eyes by this time, and was going off into a doze; but, on being pinched by the Hatter, it woke up again with a little shriek, and went on: “—t ...

#2 parent_id=126 (374 chars)
    There was a table set out under a tree in front of the house, and the March Hare and the Hatter were having tea at it: a Dormouse was sitting between them, fast asleep, a ...

#3 parent_id=137 (361 chars)
    This piece of rudeness was more than Alice could bear: she got up in great disgust, and walked off; the Dormouse fell asleep instantly, and neither of the others took the ...



### What the child index would have returned on its own, for contrast.


In [ ]:
child_hits = vectorstore.similarity_search(QUESTION, k=3)

print("what a plain child-chunk retriever would have returned:\n")
for i, doc in enumerate(child_hits, 1):
    print(f"#{i} ({len(doc.page_content)} chars): {doc.page_content[:140]}...")

avg_parent = sum(len(d.page_content) for d in results[:3]) / 3
avg_child = sum(len(d.page_content) for d in child_hits) / 3
print(f"\nmean context per result: child {avg_child:.0f} chars "
      f"vs parent {avg_parent:.0f} chars ({avg_parent / avg_child:.1f}x more)")


### 5. Three levels: the parent-splitter variant

Whole documents are sometimes too big to return - a 40-page PDF is not a useful
context unit. `ParentDocumentRetriever` accepts an optional `parent_splitter`,
which introduces a middle tier:

```
   document  ->  parent chunks (large, returned)  ->  child chunks (small, searched)
```

Use it whenever your source documents are much larger than what you want in a
prompt.

In [9]:
three_level = ParentDocumentRetriever(
    vectorstore=Chroma(collection_name="alice_3lvl", embedding_function=embeddings),
    docstore=InMemoryStore(),
    parent_splitter=RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=100),
    child_splitter=RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30),
    search_kwargs={"k": 8},
)

# Feed it a few whole chapters' worth of text so the parent splitter has work to do.
big_docs = [Document(page_content="\n\n".join(parents[i:i + 12]))
            for i in range(0, len(parents), 12)]
three_level.add_documents(big_docs)

res = three_level.invoke(QUESTION)
print(f"input documents : {len(big_docs)} "
      f"(mean {int(sum(len(d.page_content) for d in big_docs) / len(big_docs))} chars)")
print(f"returned units  : {len(res)} "
      f"(mean {int(sum(len(d.page_content) for d in res) / len(res))} chars)")
print("\ntop result:", res[0].page_content[:200], "...")

input documents : 20 (mean 4513 chars)
returned units  : 7 (mean 698 chars)

top result: He moved on as he spoke, and the Dormouse followed him: the March Hare moved into the Dormouse’s place, and Alice rather unwillingly took the place of the March Hare. The Hatter was the only one who g ...


### 6. Answer quality: the point of the exercise

Same question, same retrieval, two different context payloads.

In [10]:
def generate(docs, question):
    context = "\n\n".join(f"[{i}] {d.page_content}" for i, d in enumerate(docs))
    return ask(
        "Answer the question using ONLY the context below. Quote the detail you "
        "relied on. If the context is incomplete, say exactly what is missing.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )


print("=" * 72)
print("A) child chunks only")
print("=" * 72)
print(generate(child_hits, QUESTION))

A) child chunks only


The provided context does not contain any information about the Dormouse saying anything about a "treacle well."

**Missing Detail:** The context only mentions that the Dormouse "went on: '—that begins with an M, such as'" [0], but it does not complete the sentence or mention a treacle well.


In [11]:
print("=" * 72)
print("B) parent documents (same children matched)")
print("=" * 72)
print(generate(results[:3], QUESTION))

B) parent documents (same children matched)


The provided context does not contain any information about the Dormouse saying anything about a "treacle well." The context only mentions the Dormouse speaking about things that begin with an M, such as "mouse-traps, and the moon, and memory, and muchness" [0].

Missing detail: Any text describing the Dormouse's comments regarding a treacle well.


### 7. Pitfalls

- **Context bloat.** Three parents at 1200 characters each is 3600 characters
  where three children would have been 600. That is the trade you are making;
  keep `k` low when returning parents.
- **Duplicate parents.** The class deduplicates for you. If you hand-roll it,
  you must.
- **`k` means children, not parents.** `search_kwargs={"k": 8}` retrieves eight
  *children*; if they share parents you may get only two or three documents back.
  Set it higher than the number of parents you want.
- **The docstore is not persistent.** `InMemoryStore` disappears when the kernel
  does. Production needs `LocalFileStore`, Redis, or a database - and the
  docstore must be rebuilt in lockstep with the vector index or the ids dangle.
- **Reranking interacts with this.** Rerank the *children* (short, fits the
  cross-encoder window), then expand to parents. Reranking full parents wastes
  the window and risks truncation.

### 8. Key takeaways

- `ParentDocumentRetriever` searches child chunks and returns parent documents -
  precision of small, context of large.
- Two stores: a vector store of children carrying `doc_id`, and a key-value
  docstore of parents.
- Add a `parent_splitter` for a three-level hierarchy when source documents are
  too big to return whole.
- Deduplicate parents, and remember `k` counts children.

Next: [`03_sentence_window_retrieval.ipynb`](03_sentence_window_retrieval.ipynb),
a finer-grained variant of the same idea.